# Stage 4: Data Understanding & Data Quality Audit
**Objective:** Perform a rigorous, deep-dive data quality audit across all 9 raw Olist tables. 
We validate schema types, primary key uniqueness, null percentages, referential integrity, and chronological logic.

## Schema & Semantic Audit

### Step 1.1: Environment Setup & Directory Configuration
**Purpose:** Initialize essential data manipulation libraries (`pandas`, `pathlib`) and establish an immutable path reference pointing to the raw data repository (`data/raw/`).

In [2]:
import pandas as pd
import pathlib 

RAW_PATH = pathlib.Path("../data/raw")

print(f"loading raw dataset form:{RAW_PATH.resolve()}")

loading raw dataset form:E:\programming\end to end ml projects\olist-customer-intelligence\data\raw


### Step 1.2: Ingestion into Centralized Catalog Dictionary
**Purpose:** Ingest all 9 relational raw CSV files into an in-memory dictionary. This catalog structure enables automated programmatic audits, batch validations, and consistent operations across tables without redundant code.

In [3]:
tables = {
    "customers": pd.read_csv(RAW_PATH / "olist_customers_dataset.csv"),
    "orders": pd.read_csv(RAW_PATH / "olist_orders_dataset.csv"),
    "order_items": pd.read_csv(RAW_PATH / "olist_order_items_dataset.csv"),
    "order_payments": pd.read_csv(RAW_PATH / "olist_order_payments_dataset.csv"),
    "order_reviews": pd.read_csv(RAW_PATH / "olist_order_reviews_dataset.csv"),
    "products": pd.read_csv(RAW_PATH / "olist_products_dataset.csv"),
    "sellers": pd.read_csv(RAW_PATH / "olist_sellers_dataset.csv"),
    "geolocation": pd.read_csv(RAW_PATH / "olist_geolocation_dataset.csv"),
    "translations": pd.read_csv(RAW_PATH / "product_category_name_translation.csv")
}

for name, df in tables.items():
    print(f"Table Loaded: {name:<15} | Rows: {df.shape[0]:<8} | Cols: {df.shape[1]}")

Table Loaded: customers       | Rows: 99441    | Cols: 5
Table Loaded: orders          | Rows: 99441    | Cols: 8
Table Loaded: order_items     | Rows: 112650   | Cols: 7
Table Loaded: order_payments  | Rows: 103886   | Cols: 5
Table Loaded: order_reviews   | Rows: 99224    | Cols: 7
Table Loaded: products        | Rows: 32951    | Cols: 9
Table Loaded: sellers         | Rows: 3095     | Cols: 4
Table Loaded: geolocation     | Rows: 1000163  | Cols: 5
Table Loaded: translations    | Rows: 71       | Cols: 2


### Step 1.3: Schema & Semantic Type Audit Execution
**Purpose:** Systematically inspect every column across all ingested tables to detect semantic type mismatches:
1. **Datetime Detection:** Identify lifecycle timestamps stored as generic string (`object`) types rather than true temporal structures.
2. **Geographical Key Integrity:** Detect ZIP code prefix fields mistakenly stored as numeric types (`int64`/`float64`), which risk losing essential leading zeros.

In [4]:
schema_audit = []

for table_name, df in tables.items():
    for col in df.columns:
        dtype = str(df[col].dtype)
        sample_val = df[col].dropna().iloc[0] if not df[col].dropna().empty else None
        
        # Identification rules
        is_date_col = "date" in col or "timestamp" in col or "_at" in col
        is_zip_col = "zip_code_prefix" in col
        
        semantic_issue = "None"
        if is_date_col and dtype == "object":
            semantic_issue = "Date stored as Object (Needs Datetime conversion)"
        elif is_zip_col and dtype in ["int64", "float64"]:
            semantic_issue = "ZIP prefix stored as Numeric (Risk of losing leading zeros)"

        schema_audit.append({
            "Table": table_name,
            "Column": col,
            "Current Dtype": dtype,
            "Sample Value": str(sample_val),
            "Semantic Issue": semantic_issue
        })

df_schema_audit = pd.DataFrame(schema_audit)
print("Schema audit scan completed successfully!")

Schema audit scan completed successfully!


In [5]:
# Filter and display identified type issues
issues_found = df_schema_audit[df_schema_audit["Semantic Issue"] != "None"]
display(issues_found)

,Table,Column,Current Dtype,Sample Value,Semantic Issue
2,customers,customer_zip_code_prefix,int64,14409,ZIP prefix stored as Numeric (Risk of losing l...
8,orders,order_purchase_timestamp,object,2017-10-02 10:56:33,Date stored as Object (Needs Datetime conversion)
9,orders,order_approved_at,object,2017-10-02 11:07:15,Date stored as Object (Needs Datetime conversion)
10,orders,order_delivered_carrier_date,object,2017-10-04 19:55:00,Date stored as Object (Needs Datetime conversion)
11,orders,order_delivered_customer_date,object,2017-10-10 21:25:13,Date stored as Object (Needs Datetime conversion)
12,orders,order_estimated_delivery_date,object,2017-10-18 00:00:00,Date stored as Object (Needs Datetime conversion)
17,order_items,shipping_limit_date,object,2017-09-19 09:45:35,Date stored as Object (Needs Datetime conversion)
30,order_reviews,review_creation_date,object,2018-01-18 00:00:00,Date stored as Object (Needs Datetime conversion)
31,order_reviews,review_answer_timestamp,object,2018-01-18 21:46:59,Date stored as Object (Needs Datetime conversion)
42,sellers,seller_zip_code_prefix,int64,13023,ZIP prefix stored as Numeric (Risk of losing l...


### Step 2.1: Primary Key Uniqueness & Integrity Audit Setup
**Purpose:** Define expected primary keys (single and composite) across all entities and audit them for duplicates and missing values to prevent data multiplication during merges.

In [6]:
# Primary key definition mapping across entities
pk_definitions = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],  # Composite primary key
    "order_payments": ["order_id", "payment_sequential"],  # Composite primary key
    "order_reviews": ["review_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "geolocation": ["geolocation_zip_code_prefix"],
    "translations": ["product_category_name"]
}

pk_audit_results = []

for table_name, pk_cols in pk_definitions.items():
    df = tables[table_name]
    total_records = len(df)
    
    # Null check across PK attributes
    null_pk_count = int(df[pk_cols].isnull().any(axis=1).sum())
    
    # Duplicate check across PK combinations
    duplicate_pk_count = int(df.duplicated(subset=pk_cols, keep=False).sum())
    
    # Strict uniqueness determination
    is_unique = (duplicate_pk_count == 0) and (null_pk_count == 0)
    
    pk_audit_results.append({
        "Table": table_name,
        "Target PK / Composite Grain": " + ".join(pk_cols),
        "Total Rows": total_records,
        "Null PK Rows": null_pk_count,
        "Duplicate PK Rows": duplicate_pk_count,
        "Strictly Unique": is_unique
    })

df_pk_audit = pd.DataFrame(pk_audit_results)
print("Primary Key integrity scan completed successfully!")

Primary Key integrity scan completed successfully!


### Step 2.2: Primary Key Integrity Results Evaluation
**Purpose:** Render the primary key verification table to identify which tables have true candidate keys and which tables have high redundancy.

In [7]:
display(df_pk_audit)

,Table,Target PK / Composite Grain,Total Rows,Null PK Rows,Duplicate PK Rows,Strictly Unique
0,customers,customer_id,99441,0,0,True
1,orders,order_id,99441,0,0,True
2,order_items,order_id + order_item_id,112650,0,0,True
3,order_payments,order_id + payment_sequential,103886,0,0,True
4,order_reviews,review_id,99224,0,1603,False
5,products,product_id,32951,0,0,True
6,sellers,seller_id,3095,0,0,True
7,geolocation,geolocation_zip_code_prefix,1000163,0,999120,False
8,translations,product_category_name,71,0,0,True


### Step 3.1: Systemic Missing Value & Completeness Audit Execution
**Purpose:** Scan all 9 relational tables across every column to identify missing value distributions, calculate exact null counts and missing percentages, and assess business reasons behind non-available records.

In [8]:
completeness_audit = []

for table_name, df in tables.items():
    total_records = len(df)
    for col in df.columns:
        null_count = int(df[col].isnull().sum())
        if null_count > 0:
            null_pct = round((null_count / total_records) * 100, 3)
            completeness_audit.append({
                "Table": table_name,
                "Column": col,
                "Data Type": str(df[col].dtype),
                "Missing Rows": null_count,
                "Total Rows": total_records,
                "Missing Percentage (%)": null_pct
            })

df_completeness_audit = pd.DataFrame(completeness_audit)
print("Completeness & null profiling scan completed successfully!")

Completeness & null profiling scan completed successfully!


### Step 3.2: Missing Value Profiling Results Evaluation
**Purpose:** Render the ranked missingness report sorted descending by missing percentage to isolate critical features requiring business imputation vs those eligible for feature pruning.

In [9]:
if df_completeness_audit.empty:
    print("Zero missing values detected across the entire database.")
else:
    # Sort descending by missing percentage for visibility
    display(df_completeness_audit.sort_values(by="Missing Percentage (%)", ascending=False).reset_index(drop=True))

,Table,Column,Data Type,Missing Rows,Total Rows,Missing Percentage (%)
0,order_reviews,review_comment_title,object,87656,99224,88.342
1,order_reviews,review_comment_message,object,58247,99224,58.703
2,orders,order_delivered_customer_date,object,2965,99441,2.982
3,products,product_name_lenght,float64,610,32951,1.851
4,products,product_category_name,object,610,32951,1.851
5,products,product_description_lenght,float64,610,32951,1.851
6,products,product_photos_qty,float64,610,32951,1.851
7,orders,order_delivered_carrier_date,object,1783,99441,1.793
8,orders,order_approved_at,object,160,99441,0.161
9,products,product_weight_g,float64,2,32951,0.006


### Step 4.1: Domain Constraints & Value Range Anomaly Detection
**Purpose:** Verify mathematical and business domain integrity across numerical attributes. Inspect prices, freight costs, payment amounts, review ratings, and product physical metrics for non-positive, out-of-bounds, or illogical values.

In [14]:
range_anomalies = []

# 1. order items (prices & freights)

df_items = tables["order_items"]
zero_neg_price = (df_items["price"]<=0).sum()
zero_neg_freight = (df_items["freight_value"]<0).sum()

range_anomalies.append({
    "Table": "order_items",
    "Attribute": "price",
    'Condition Checked': "price <= 0",
    "Violation Count": int(zero_neg_price),
    "Min Found": float(df_items["price"].min()),
    "Max Found": float(df_items["price"].max())
})

range_anomalies.append({
    "Table": "order_items",
    "Attribute": "freight",
    'Condition Checked': "price < 0",
    "Violation Count": int(zero_neg_freight),
    "Min Found": float(df_items["freight_value"].min()),
    "Max Found": float(df_items["freight_value"].max())

})

# 2. Order Reviews (Ratings 1 to 5)
df_reviews = tables["order_reviews"]
invalid_reviews = (~df_reviews["review_score"].between(1, 5)).sum()

range_anomalies.append({
    "Table": "order_reviews",
    "Attribute": "review_score",
    "Condition Checked": "Score not in [1, 5]",
    "Violation Count": int(invalid_reviews),
    "Min Found": float(df_reviews["review_score"].min()),
    "Max Found": float(df_reviews["review_score"].max())
})

# 3. Order Payments (Values & Installments)
df_payments = tables["order_payments"]
zero_neg_payment = (df_payments["payment_value"] <= 0).sum()
zero_installments = (df_payments["payment_installments"] < 0).sum()

range_anomalies.append({
    "Table": "order_payments",
    "Attribute": "payment_value",
    "Condition Checked": "Payment Value <= 0",
    "Violation Count": int(zero_neg_payment),
    "Min Found": float(df_payments["payment_value"].min()),
    "Max Found": float(df_payments["payment_value"].max())
})
range_anomalies.append({
    "Table": "order_payments",
    "Attribute": "payment_installments",
    "Condition Checked": "Installments < 0",
    "Violation Count": int(zero_installments),
    "Min Found": float(df_payments["payment_installments"].min()),
    "Max Found": float(df_payments["payment_installments"].max())
})

# 4. Products (Physical Metrics)
df_products = tables["products"]
zero_weight = (df_products["product_weight_g"] <= 0).sum()

range_anomalies.append({
    "Table": "products",
    "Attribute": "product_weight_g",
    "Condition Checked": "Weight <= 0",
    "Violation Count": int(zero_weight),
    "Min Found": float(df_products["product_weight_g"].min()),
    "Max Found": float(df_products["product_weight_g"].max())
})

df_range_audit = pd.DataFrame(range_anomalies)
print("Domain & value range scan completed successfully!")

display(df_range_audit)

Domain & value range scan completed successfully!


,Table,Attribute,Condition Checked,Violation Count,Min Found,Max Found
0,order_items,price,price <= 0,0,0.85,6735.00
1,order_items,freight,price < 0,0,0.00,409.68
2,order_reviews,review_score,"Score not in [1, 5]",0,1.00,5.00
3,order_payments,payment_value,Payment Value <= 0,9,0.00,13664.08
4,order_payments,payment_installments,Installments < 0,0,0.00,24.00
5,products,product_weight_g,Weight <= 0,4,0.00,40425.00


### Step 5.1: Referential Integrity & Orphan Record Audit Execution
**Purpose:** Test foreign key relationships across interconnected tables. Identify orphan child records that lack a valid parent entity reference to prevent data loss or Cartesian explosion during inner joins.

In [ ]:
# fk_audit_results = []

# # 1. Orders -> Customers (Does every order map to a valid customer?)
# orphaned_orders = (~tables["orders"]["customer_id"].isin(tables["customers"]["customer_id"])).sum()
# fk_audit_results.append({
#     "Child Table": "orders",
#     "Child Foreign Key": "customer_id",
#     "Parent Table": "customers",
#     "Parent Key": "customer_id",
#     "Orphan Record Count": int(orphaned_orders)
# })

# # 2. Order Items -> Orders
# orphaned_items_orders = (~tables["order_items"]["order_id"].isin(tables["orders"]["order_id"])).sum()
# fk_audit_results.append({
#     "Child Table": "order_items",
#     "Child Foreign Key": "order_id",
#     "Parent Table": "orders",
#     "Parent Key": "order_id",
#     "Orphan Record Count": int(orphaned_items_orders)
# })

# # 3. Order Items -> Products
# orphaned_items_products = (~tables["order_items"]["product_id"].isin(tables["products"]["product_id"])).sum()
# fk_audit_results.append({
#     "Child Table": "order_items",
#     "Child Foreign Key": "product_id",
#     "Parent Table": "products",
#     "Parent Key": "product_id",
#     "Orphan Record Count": int(orphaned_items_products)
# })

# # 4. Order Items -> Sellers
# orphaned_items_sellers = (~tables["order_items"]["seller_id"].isin(tables["sellers"]["seller_id"])).sum()
# fk_audit_results.append({
#     "Child Table": "order_items",
#     "Child Foreign Key": "seller_id",
#     "Parent Table": "sellers",
#     "Parent Key": "seller_id",
#     "Orphan Record Count": int(orphaned_items_sellers)
# })

# # 5. Order Payments -> Orders
# orphaned_payments = (~tables["order_payments"]["order_id"].isin(tables["orders"]["order_id"])).sum()
# fk_audit_results.append({
#     "Child Table": "order_payments",
#     "Child Foreign Key": "order_id",
#     "Parent Table": "orders",
#     "Parent Key": "order_id",
#     "Orphan Record Count": int(orphaned_payments)
# })

# # 6. Order Reviews -> Orders
# orphaned_reviews = (~tables["order_reviews"]["order_id"].isin(tables["orders"]["order_id"])).sum()
# fk_audit_results.append({
#     "Child Table": "order_reviews",
#     "Child Foreign Key": "order_id",
#     "Parent Table": "orders",
#     "Parent Key": "order_id",
#     "Orphan Record Count": int(orphaned_reviews)
# })

# df_fk_audit = pd.DataFrame(fk_audit_results)
# print("Referential integrity scan completed successfully!")

Referential integrity scan completed successfully!


*oop version*

In [17]:

class ForeignKeyAuditor:

    def __init__(self, tables):
        self.tables = tables
        self.results = []

    def audit_relationship(self, child_table, child_key,
                           parent_table, parent_key):

        child_df = self.tables[child_table]
        parent_df = self.tables[parent_table]

        orphan_count = (
            ~child_df[child_key].isin(parent_df[parent_key])
        ).sum()

        self.results.append({
            "Child Table": child_table,
            "Child Foreign Key": child_key,
            "Parent Table": parent_table,
            "Parent Key": parent_key,
            "Orphan Record Count": int(orphan_count)
        })

    def run_audit(self):

        relationships = [
            ("orders", "customer_id", "customers", "customer_id"),
            ("order_items", "order_id", "orders", "order_id"),
            ("order_items", "product_id", "products", "product_id"),
            ("order_items", "seller_id", "sellers", "seller_id"),
            ("order_payments", "order_id", "orders", "order_id"),
            ("order_reviews", "order_id", "orders", "order_id")
        ]

        for relationship in relationships:
            self.audit_relationship(*relationship)

        return pd.DataFrame(self.results)


# Create auditor object
auditor = ForeignKeyAuditor(tables)

# Run audit
df_fk_audit = auditor.run_audit()

print("Referential integrity scan completed successfully!")

display(df_fk_audit)

Referential integrity scan completed successfully!


,Child Table,Child Foreign Key,Parent Table,Parent Key,Orphan Record Count
0,orders,customer_id,customers,customer_id,0
1,order_items,order_id,orders,order_id,0
2,order_items,product_id,products,product_id,0
3,order_items,seller_id,sellers,seller_id,0
4,order_payments,order_id,orders,order_id,0
5,order_reviews,order_id,orders,order_id,0


### Step 6.1: Temporal Consistency & Chronological Logic Audit
**Purpose:** Convert order timestamps temporarily into datetime format to validate the chronological sequence of the e-commerce lifecycle. Detect date anomalies such as deliveries logged before order placement or carrier handovers recorded after customer delivery.

In [18]:
# Create temporary datetime series for orders to perform chronological arithmetic
df_orders = tables["orders"].copy()

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    df_orders[col] = pd.to_datetime(df_orders[col], errors="coerce")

# Chronological validations
# 1. Approval before purchase?
approval_before_purchase = (df_orders["order_approved_at"] < df_orders["order_purchase_timestamp"]).sum()

# 2. Delivered to customer before purchase?
delivered_before_purchase = (df_orders["order_delivered_customer_date"] < df_orders["order_purchase_timestamp"]).sum()

# 3. Carrier handover after customer delivery?
carrier_after_delivery = (df_orders["order_delivered_carrier_date"] > df_orders["order_delivered_customer_date"]).sum()

# 4. Total orders delivered after estimated date (Late Deliveries - Business SLA Breaches)
delivered_mask = df_orders["order_delivered_customer_date"].notnull()
late_deliveries = (df_orders.loc[delivered_mask, "order_delivered_customer_date"] > df_orders.loc[delivered_mask, "order_estimated_delivery_date"]).sum()
late_pct = round((late_deliveries / delivered_mask.sum()) * 100, 2)

temporal_audit = [
    {
        "Lifecycle Rule": "Approved < Purchased",
        "Anomaly Type": "System / Logging Error",
        "Violation Count": int(approval_before_purchase),
        "Business Action": "Flag as chronological defect"
    },
    {
        "Lifecycle Rule": "Delivered to Customer < Purchased",
        "Anomaly Type": "Critical Data Corruption",
        "Violation Count": int(delivered_before_purchase),
        "Business Action": "Must drop or investigate"
    },
    {
        "Lifecycle Rule": "Carrier Handover > Customer Delivery",
        "Anomaly Type": "Logistics Sequence Conflict",
        "Violation Count": int(carrier_after_delivery),
        "Business Action": "Reconcile delivery timestamps"
    },
    {
        "Lifecycle Rule": "Delivered > Estimated Date",
        "Anomaly Type": "Legitimate Business SLA Breach (Late)",
        "Violation Count": int(late_deliveries),
        "Business Action": f"Key ML Target Feature ({late_pct}% SLA breaches)"
    }
]

df_temporal_audit = pd.DataFrame(temporal_audit)
print("Temporal consistency scan completed successfully!")

display(df_temporal_audit)

Temporal consistency scan completed successfully!


,Lifecycle Rule,Anomaly Type,Violation Count,Business Action
0,Approved < Purchased,System / Logging Error,0,Flag as chronological defect
1,Delivered to Customer < Purchased,Critical Data Corruption,0,Must drop or investigate
2,Carrier Handover > Customer Delivery,Logistics Sequence Conflict,23,Reconcile delivery timestamps
3,Delivered > Estimated Date,Legitimate Business SLA Breach (Late),7827,Key ML Target Feature (8.11% SLA breaches)
